## **Problem Set 4.**  Text III: attention & transformers
### *Goal: understand and reproduce the attention mechanism, then use a pretrained transformer.*

In [1]:
# Setup
import torch
import torch.nn.functional as F
import math
import torch.nn as nn

torch.manual_seed(0)

#### 1. **[Theory]** Derive scaled dot-product attention $\text{Attn}(Q, K, V) = \text{softmax}(QK^\top / \sqrt{d_k})V$. Give the variance argument for the $1/\sqrt{d_k}$ scaling (what happens to the softmax as $d_k$ grows without it?).

#### 2. **[Theory]** Show that self-attention is permutation-equivariant, i.e. permuting the input rows permutes the output rows identically. Explain why this forces the use of positional encodings, and analyse the sinusoidal encoding's ability to represent relative positions.

#### 3. **[Theory]** Count the parameters and the asymptotic compute of one multi-head attention layer with $h$ heads and model dimension $d$, and compare with a dense layer of the same width. Justify the $O(n^2d)$ cost in sequence length $n$.

#### 4. **[Python]** Implement scaled dot-product attention and multi-head attention from scratch and check your output against `torch.nn.MultiheadAttention` on random inputs (max abs. error $< 10^{-5}$).

1. [Implement Scaled Dot-Product Attention](https://apxml.com/courses/foundations-transformers-architecture/chapter-2-attention-mechanism-core-concepts/practice-implementing-attention)

Standard attention formula:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right)V$$

where:
- ($Q$) - queries 
- ($K$) - keys 
- ($V$) - values
- ($d_k$) - key dimension


2. [Implement Multi-Head Attention (MHA)](https://inamdaraditya.medium.com/understanding-multi-head-attention-a-deep-dive-with-code-0d414c073154)

Class (inheriting from torch.nn.Module) that:
- Splits the inputs into a specified number of "heads".
- Runs the attention calculation in parallel for each head.
- Concatenates the outputs of individual heads and passes them through a final linear projection layer.

In [2]:
# Scaled Dot-Product Attention
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Computes Scaled Dot-Product Attention.

    Args:
        query: Query tensor; shape (batch_size, seq_len_q, d_k)
        key: Tensor; shape (batch_size, seq_len_k, d_k)
        value: Value tensor; shape (batch_size, seq_len_k, d_v)
               Note: seq_len_k must match between key and value.
               d_v (value dimension) can differ from d_k (query dimension).
        mask: Optional mask tensor; shape broadcastable to (batch_size, seq_len_q, seq_len_k).
              Positions with True or 1 indicate values to keep, False or 0 to mask out.

    Returns:
        output: The attention output tensor; shape (batch_size, seq_len_q, d_v)
        attention_weights: The attention weights; shape (batch_size, seq_len_q, seq_len_k)
    """
    # Dimension of keys
    d_k = key.size(-1)

    # 1. Calculate QK^T
    # (batch_size, seq_len_q, d_k) @ (batch_size, d_k, seq_len_k) -> (batch_size, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2, -1))

    # 2. Scale by sqrt(d_k)
    scores = scores / math.sqrt(d_k)

    # 3. Apply mask (if provided)
    if mask is not None:
        # Mask values are typically False for positions to mask out
        # We need to set masked positions to a large negative value (-inf) before softmax
        # Use fill_value=-1e9 or similar large negative number for numerical stability
        scores = scores.masked_fill(mask == 0, -1e9) # PyTorch convention: 0/False means mask

    # 4. Apply softmax to get attention weights
    # Softmax is applied on the last dimension (seq_len_k)
    attention_weights = F.softmax(scores, dim=-1)

    # Handle potential NaN from softmax if all scores in a row are -inf
    # This can happen if a query position is masked entirely against all keys.
    # Replace NaN with 0.
    attention_weights = torch.nan_to_num(attention_weights)

    # 5. Multiply weights by V
    # (batch_size, seq_len_q, seq_len_k) @ (batch_size, seq_len_k, d_v) -> (batch_size, seq_len_q, d_v)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights


In [3]:
# Multi-Head Attention (MHA)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Linear projections for Q, K, V
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        # Output linear layer
        self.out_linear = nn.Linear(d_model, d_model)
    
    def forward(self, x: torch.Tensor, mask=None):
        B, T, D = x.size()
        
        # Linear projections
        Q = self.q_linear(x)
        K = self.k_linear(x)
        V = self.v_linear(x)
        
        # Split into heads
        Q = Q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Compute attention scores
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1)
        
        # Apply attention weights to V
        out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        
        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        
        # Final linear layer
        return self.out_linear(out)

In [ ]:
# Verification of error < 10^-5
d_model = 64
n_heads = 4
batch_size = 2
seq_len = 10

# Create random input
x = torch.randn(batch_size, seq_len, d_model)

# Initialize both models
my_mha = MultiHeadAttention(d_model, n_heads)
torch_mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)

with torch.no_grad():
    weight_q, weight_k, weight_v = torch.chunk(torch_mha.in_proj_weight, 3, dim=0)
    my_mha.q_linear.weight.copy_(weight_q)
    my_mha.k_linear.weight.copy_(weight_k)
    my_mha.v_linear.weight.copy_(weight_v)
    
    bias_q, bias_k, bias_v = torch.chunk(torch_mha.in_proj_bias, 3, dim=0)
    my_mha.q_linear.bias.copy_(bias_q)
    my_mha.k_linear.bias.copy_(bias_k)
    my_mha.v_linear.bias.copy_(bias_v)
    
    my_mha.out_linear.weight.copy_(torch_mha.out_proj.weight)
    my_mha.out_linear.bias.copy_(torch_mha.out_proj.bias)

# Run both models on the same input
my_output = my_mha(x)
# nn.MultiheadAttention returns a tuple (output, attn_weights)
torch_output, _ = torch_mha(x, x, x) 

# Check the maximum absolute error
max_error = torch.max(torch.abs(my_output - torch_output)).item()
print(f"Maximum absolute error: {max_error}")

assert max_error < 1e-5, "The error is too large"
print("Test passed successfully")

Maximum absolute error: 1.1920928955078125e-07
Test passed successfully


#### 5. **[Python]** Use a pretrained HuggingFace transformer two ways: (a) as a frozen sentence-embedding extractor feeding a linear classifier, and (b) fine-tuned end-to-end. Compare both against the TF-IDF baseline from Problem Set 3 on the same data.

#### 6. **[Analysis]** Visualise the attention weights of one head for two example sentences and discuss whether the pattern is interpretable. Note at least one caveat about reading meaning into attention maps.